In [1]:
import pandas as pd

from lib.preprocess import preprocess
from lib.config import ExperimentConfig
from lib.experiments import run_grid_search, extract_model_params

df = pd.read_csv("dsat.csv", dtype={352: "str", 353: "str"})
(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
    X_train_full,
    y_train_full,
    features,
) = preprocess(df, feature_set_num=1)

In [2]:
base_cb_cfg = ExperimentConfig(
    name="catboost_grid",
    model_type="catboost",
    params={
        "iterations": 1000,
        "random_state": 42,
        "verbose": False,
    },
)

In [3]:
param_grid = {
    "model__depth": [4, 6, 8],
    "model__learning_rate": [0.01, 0.03],
    "model__l2_leaf_reg": [1, 3, 5],
}

In [ ]:
grid_cb = run_grid_search(
    base_cfg=base_cb_cfg,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=1
)

print("Best params (raw):", grid_cb.best_params_)
print("Best CV RMSE:", -grid_cb.best_score_)

cb_model_params = extract_model_params(grid_cb.best_params_)
cb_model_params

In [ ]:
base_rf_cfg = ExperimentConfig(
    name="rf_grid",
    model_type="rf",
    params={
        "n_estimators": 200,
        "random_state": 42,
        "n_jobs": -1,
    },
    scaler=None,
)

In [ ]:
param_grid = {
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"],
}

In [ ]:
grid_rf = run_grid_search(
    base_cfg=base_rf_cfg,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    scoring="neg_root_mean_squared_error",
    cv=3,
)

print("Best params (raw):", grid_rf.best_params_)
print("Best CV RMSE:", -grid_rf.best_score_)

In [ ]:
rf_model_params = extract_model_params(grid_rf.best_params_)
rf_model_params